In [ ]:
# Standalone S5 sweep aggregation cell
# CPU-only: checkpoint/model yüklemez, yalnızca JSON summary dosyalarını okur.

from pathlib import Path
import json
import math
import numpy as np
import pandas as pd


SAVE_DIR = Path("/content/drive/MyDrive/Grokking_S5_Sweep_v2")

if not SAVE_DIR.exists():
    raise FileNotFoundError(
        f"S5 sonuç klasörü bulunamadı: {SAVE_DIR}\n"
        "Drive mount edilmiş mi ve klasör adı doğru mu kontrol et."
    )

# ---------------------------------------------------------------------
# 1. Bütün JSON dosyalarını oku
# ---------------------------------------------------------------------

records = []
errors = []

for path in SAVE_DIR.rglob("*.json"):
    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        # Notebook kayıtları çoğunlukla {"summary": {...}} biçiminde.
        # Bazı dosyalar doğrudan summary dict olabilir.
        if isinstance(obj, dict) and isinstance(obj.get("summary"), dict):
            row = dict(obj["summary"])
        elif isinstance(obj, dict) and (
            "config" in obj or "seed" in obj or "grok_epoch_99" in obj
        ):
            row = dict(obj)
        else:
            continue

        # Eksik metadata'yı gerektiğinde üst seviyeden tamamla.
        if isinstance(obj, dict):
            for key in [
                "config", "seed", "model_type", "bounded_head",
                "uniform_attention", "use_positional_embedding",
                "wd", "lr", "tau", "d_model", "mlp_dim",
                "layers", "max_epochs", "status"
            ]:
                if row.get(key) is None and obj.get(key) is not None:
                    row[key] = obj[key]

        row["_path"] = str(path)
        row["_mtime"] = path.stat().st_mtime
        records.append(row)

    except Exception as exc:
        errors.append((str(path), repr(exc)))

if not records:
    raise RuntimeError(
        f"{SAVE_DIR} altında okunabilir bir summary JSON bulunamadı."
    )

df = pd.DataFrame(records)

# ---------------------------------------------------------------------
# 2. Kolonları ve tipleri temizle
# ---------------------------------------------------------------------

numeric_cols = [
    "seed", "wd", "lr", "tau", "d_model", "mlp_dim", "layers",
    "max_epochs", "last_logged_epoch",
    "train_epoch_95", "train_epoch_99",
    "grok_epoch_95", "grok_epoch_99",
    "gap_95", "gap_99",
    "peak_test_acc", "final_train_acc", "final_test_acc"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Config adı eksikse parent klasörü kullan.
if "config" not in df.columns:
    df["config"] = None

df["config"] = df.apply(
    lambda r: (
        str(r["config"])
        if pd.notna(r.get("config")) and str(r.get("config")).strip()
        else Path(r["_path"]).parent.name
    ),
    axis=1
)

# Seed eksikse dosya adından çıkarmaya çalış.
if "seed" not in df.columns:
    df["seed"] = np.nan

missing_seed = df["seed"].isna()
df.loc[missing_seed, "seed"] = (
    df.loc[missing_seed, "_path"]
      .str.extract(r"seed[_-]?(\d+)", expand=False)
      .pipe(pd.to_numeric, errors="coerce")
)

# Bazı eski özetlerde gap kaydedilmemiş olabilir.
if "gap_99" not in df.columns:
    df["gap_99"] = np.nan

if "grok_epoch_99" not in df.columns:
    df["grok_epoch_99"] = np.nan

if "train_epoch_99" not in df.columns:
    df["train_epoch_99"] = np.nan

missing_gap99 = (
    df["gap_99"].isna()
    & df["grok_epoch_99"].notna()
    & df["train_epoch_99"].notna()
)
df.loc[missing_gap99, "gap_99"] = (
    df.loc[missing_gap99, "grok_epoch_99"]
    - df.loc[missing_gap99, "train_epoch_99"]
)

# Aynı config + seed birden fazla JSON'da varsa en yeni dosyayı tut.
# Seed bulunamayan kayıtları path ile ayrı tutuyoruz.
df["_dedupe_seed"] = df["seed"].where(
    df["seed"].notna(),
    df["_path"].map(lambda x: f"path:{x}")
)

df = (
    df.sort_values("_mtime")
      .drop_duplicates(["config", "_dedupe_seed"], keep="last")
      .reset_index(drop=True)
)

# Başarı göstergeleri
df["reached_95"] = df.get(
    "grok_epoch_95", pd.Series(np.nan, index=df.index)
).notna()

df["reached_99"] = df["grok_epoch_99"].notna()

# Accuracy değerleri 0–1 kabul ediliyor.
df["peak_test_pct"] = (
    100 * df.get("peak_test_acc", pd.Series(np.nan, index=df.index))
)

# Gerçekte tamamlanan horizon:
# max_epochs varsa onu, yoksa last_logged_epoch'u kullan.
df["horizon"] = df.get(
    "max_epochs", pd.Series(np.nan, index=df.index)
)

if "last_logged_epoch" in df.columns:
    df["horizon"] = df["horizon"].fillna(df["last_logged_epoch"])

# ---------------------------------------------------------------------
# 3. Config-level özet
# ---------------------------------------------------------------------

def successful_mean(series, success_mask):
    vals = pd.to_numeric(series[success_mask], errors="coerce").dropna()
    return vals.mean() if len(vals) else np.nan

def successful_std(series, success_mask):
    vals = pd.to_numeric(series[success_mask], errors="coerce").dropna()
    return vals.std(ddof=1) if len(vals) > 1 else np.nan

summary_rows = []

for config, g in df.groupby("config", sort=True):
    success99 = g["reached_99"]

    def first_non_null(col):
        if col not in g.columns:
            return np.nan
        vals = g[col].dropna()
        return vals.iloc[0] if len(vals) else np.nan

    peak_vals = g["peak_test_pct"].dropna()

    summary_rows.append({
        "config": config,
        "residual": first_non_null("model_type"),
        "bounded_head": first_non_null("bounded_head"),
        "uniform_attention": first_non_null("uniform_attention"),
        "positions": first_non_null("use_positional_embedding"),
        "wd": first_non_null("wd"),
        "lr": first_non_null("lr"),
        "tau": first_non_null("tau"),
        "d_model": first_non_null("d_model"),
        "layers": first_non_null("layers"),
        "horizon": g["horizon"].max(),
        "n": len(g),
        "reached_99_n": int(success99.sum()),
        "reached_99": f"{int(success99.sum())}/{len(g)}",
        "mean_test_99": successful_mean(g["grok_epoch_99"], success99),
        "std_test_99": successful_std(g["grok_epoch_99"], success99),
        "median_test_99": (
            g.loc[success99, "grok_epoch_99"].median()
            if success99.any() else np.nan
        ),
        "mean_gap_99": successful_mean(g["gap_99"], success99),
        "std_gap_99": successful_std(g["gap_99"], success99),
        "mean_peak_pct": peak_vals.mean() if len(peak_vals) else np.nan,
        "min_peak_pct": peak_vals.min() if len(peak_vals) else np.nan,
        "max_peak_pct": peak_vals.max() if len(peak_vals) else np.nan,
    })

summary = pd.DataFrame(summary_rows)

# ---------------------------------------------------------------------
# 4. Okunabilir biçimlendirme
# ---------------------------------------------------------------------

def fmt_int(x):
    return "NA" if pd.isna(x) else f"{int(round(x)):,}"

def fmt_mean_std(mean, std):
    if pd.isna(mean):
        return "NA"
    if pd.isna(std):
        return fmt_int(mean)
    return f"{int(round(mean)):,} ± {int(round(std)):,}"

def fmt_lr(x):
    if pd.isna(x):
        return "NA"
    return f"{x:g}"

def fmt_tau(x):
    if pd.isna(x):
        return "raw"
    return f"{x:g}"

def fmt_peak_range(row):
    lo, hi = row["min_peak_pct"], row["max_peak_pct"]
    if pd.isna(lo) or pd.isna(hi):
        return "NA"
    if abs(lo - hi) < 0.05:
        return f"{lo:.2f}%"
    return f"{lo:.2f}–{hi:.2f}%"

summary["Test-99"] = summary.apply(
    lambda r: fmt_mean_std(r["mean_test_99"], r["std_test_99"])
    if r["reached_99_n"] > 0
    else f"fail, peak {fmt_peak_range(r)}",
    axis=1
)

summary["Delta_99"] = summary.apply(
    lambda r: fmt_mean_std(r["mean_gap_99"], r["std_gap_99"])
    if r["reached_99_n"] > 0
    else "NA",
    axis=1
)

summary["LR"] = summary["lr"].map(fmt_lr)
summary["Tau"] = summary["tau"].map(fmt_tau)
summary["Horizon"] = summary["horizon"].map(fmt_int)

full_table = summary[[
    "config", "residual", "bounded_head", "wd", "LR", "Tau",
    "d_model", "layers", "Horizon", "n",
    "Test-99", "Delta_99", "reached_99"
]].sort_values(
    ["residual", "bounded_head", "wd", "LR", "config"],
    na_position="last"
)

print("\nFULL S5 CONFIGURATION SUMMARY\n")
print(full_table.to_string(index=False))

# ---------------------------------------------------------------------
# 5. Q4 için ilgili koşulları seç
# ---------------------------------------------------------------------

# Q4'ün erken durdurma ve hyperparameter robustness argümanına
# doğrudan hizmet eden config'ler.
Q4_CONFIGS = [
    "layernorm_learned_control",
    "layernorm_bounded_wd1_control",
    "sphere_wd1_corrected",
    "sphere_raw_wd1_control",
    "sphere_nowd_main",
    "sphere_nowd_long_200k",
    "sphere_nowd_lr_3e-4",
    "sphere_nowd_lr_3e-3",
    "sphere_nowd_tau_5",
    "sphere_nowd_tau_20",
    "sphere_nowd_wide_256",
    "sphere_nowd_deep_2layer",
]

q4 = summary[summary["config"].isin(Q4_CONFIGS)].copy()

# Listede olup Drive'da bulunmayan config'leri bildir.
missing_configs = [c for c in Q4_CONFIGS if c not in set(summary["config"])]

q4["Configuration"] = q4["config"].replace({
    "layernorm_learned_control": "LayerNorm, raw",
    "layernorm_bounded_wd1_control": "LayerNorm, bounded",
    "sphere_wd1_corrected": "Sphere, bounded",
    "sphere_raw_wd1_control": "Sphere, raw",
    "sphere_nowd_main": "Sphere, bounded",
    "sphere_nowd_long_200k": "Sphere, bounded, long run",
    "sphere_nowd_lr_3e-4": "Sphere, bounded, LR=3e-4",
    "sphere_nowd_lr_3e-3": "Sphere, bounded, LR=3e-3",
    "sphere_nowd_tau_5": "Sphere, bounded, tau=5",
    "sphere_nowd_tau_20": "Sphere, bounded, tau=20",
    "sphere_nowd_wide_256": "Sphere, bounded, width=256",
    "sphere_nowd_deep_2layer": "Sphere, bounded, depth=2",
})

q4_table = q4[[
    "Configuration", "wd", "LR", "Horizon", "n",
    "Test-99", "Delta_99", "reached_99"
]].rename(columns={
    "wd": "WD",
    "reached_99": "Reached 99%",
})

# İstenen mantıksal sırayı koru.
q4_table["_order"] = q4_table["Configuration"].map({
    "LayerNorm, raw": 0,
    "LayerNorm, bounded": 1,
    "Sphere, bounded": 2,
    "Sphere, raw": 3,
    "Sphere, bounded, long run": 4,
    "Sphere, bounded, LR=3e-4": 5,
    "Sphere, bounded, LR=3e-3": 6,
    "Sphere, bounded, tau=5": 7,
    "Sphere, bounded, tau=20": 8,
    "Sphere, bounded, width=256": 9,
    "Sphere, bounded, depth=2": 10,
}).fillna(999)

q4_table = (
    q4_table.sort_values(["_order", "WD"], ascending=[True, False])
            .drop(columns="_order")
)

print("\n\nQ4 COMPACT TABLE\n")
print(q4_table.to_string(index=False))

print("\n\nQ4 MARKDOWN TABLE\n")
print(q4_table.to_markdown(index=False))

# ---------------------------------------------------------------------
# 6. Per-seed doğrulama tablosu
# ---------------------------------------------------------------------

per_seed_cols = [
    "config", "seed", "wd", "lr", "tau", "d_model", "layers",
    "horizon", "train_epoch_99", "grok_epoch_99", "gap_99",
    "peak_test_pct", "final_train_acc", "final_test_acc", "_path"
]
per_seed_cols = [c for c in per_seed_cols if c in df.columns]

per_seed = df[per_seed_cols].sort_values(["config", "seed"])

print("\n\nPER-SEED RESULTS\n")
print(per_seed.to_string(index=False))

# CSV çıktıları
summary_csv = SAVE_DIR / "rebuttal_s5_configuration_summary.csv"
q4_csv = SAVE_DIR / "rebuttal_s5_q4_table.csv"
seed_csv = SAVE_DIR / "rebuttal_s5_per_seed.csv"

summary.to_csv(summary_csv, index=False)
q4_table.to_csv(q4_csv, index=False)
per_seed.to_csv(seed_csv, index=False)

print("\nSaved:")
print(summary_csv)
print(q4_csv)
print(seed_csv)

if missing_configs:
    print("\nWARNING: These expected configs were not found:")
    for name in missing_configs:
        print(" ", name)

if errors:
    print(f"\nWARNING: {len(errors)} JSON file(s) could not be read.")
    for path, err in errors[:10]:
        print(" ", path, err)